## Section 1. Load and Clean Data
---
Block 1-1. Load Data:
- This block is used to load data and convert time-series data into tabular data (harmonic components).

Block 1-2. Basic Data Visualization:
- This block is used to visualize data. 
---

In [ ]:
import torch
import numpy as np
import random
import os

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # For deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(0)

In [ ]:
# =========================================================================================
# Block 1-1: Load Data
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

# Read the waveforms from the CSV file "B_waveform[T].csv"
# The file is expected to have shape (N, seq_len)
waveforms = np.loadtxt("3C90/B_waveform[T].csv", delimiter=",")

# Basic data analysis
num_waveforms, seq_len = waveforms.shape
print(f"Number of waveforms: {num_waveforms}")
print(f"Sequence length: {seq_len}")

# Parameter: number of harmonics to extract (excluding DC)
num_harmonics = 13  # Convert the time-series data into 13 harmonic components

# Load additional input variables
frequencies = np.loadtxt("3C90/Frequency[Hz].csv", delimiter=",")  # shape: (num_waveforms, 1)
temperatures = np.loadtxt("3C90/Temperature[C].csv", delimiter=",")  # shape: (num_waveforms, 1)

# Ensure shapes are (N, 1)
if frequencies.ndim == 1:
    frequencies = frequencies.reshape(-1, 1)
if temperatures.ndim == 1:
    temperatures = temperatures.reshape(-1, 1)

# Compute the FFT for each waveform using the corresponding frequency for each sample
# The frequency in "Frequency[Hz].csv" is the fundamental frequency for each waveform
harmonic_magnitudes = np.zeros((waveforms.shape[0], num_harmonics))

for i in range(waveforms.shape[0]):
    waveform = waveforms[i]
    freq = frequencies[i, 0]  # fundamental frequency in Hz
    n = len(waveform)
    # FFT
    fft_result = np.fft.rfft(waveform)
    fft_magnitude = np.abs(fft_result) / n * 2  # scale for amplitude (except DC)
    # Frequency bins for this waveform
    sample_rate = freq * n  # If one period is sampled, sample_rate = freq * n / period_length, but assume n samples per period
    freqs = np.fft.rfftfreq(n, d=1.0/(freq * n))
    # The k-th harmonic is at k * fundamental frequency
    for k in range(1, num_harmonics + 1):
        harmonic_freq = k * freq
        # Find the closest bin to the harmonic frequency
        idx = np.argmin(np.abs(freqs - harmonic_freq))
        harmonic_magnitudes[i, k-1] = fft_magnitude[idx]

# Load output prediction variable
volumetric_losses = np.loadtxt("3C90/Volumetric_losses[Wm-3].csv", delimiter=",")  # shape: (num_waveforms, 1)
if volumetric_losses.ndim == 1:
    volumetric_losses = volumetric_losses.reshape(-1, 1)

# Prepare the input feature matrix: [harmonic magnitudes | frequency | temperature]
X_full = np.hstack([harmonic_magnitudes, frequencies, temperatures])  # shape: (num_waveforms, num_harmonics+2)
y_full = volumetric_losses  # shape: (num_waveforms, 1)

# Downsample to 10% with stratification to preserve the distribution of y
# We'll stratify on binned log(y) to preserve the output distribution
epsilon = 1e-12
y_log_full = np.log(y_full.flatten() + epsilon)
num_bins = 20
y_bins = np.digitize(y_log_full, np.histogram(y_log_full, bins=num_bins)[1][1:-1])

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.75, random_state=42)
for down_idx, _ in sss.split(X_full, y_bins):
    X = X_full[down_idx]
    y = y_full[down_idx]
    y_log = y_log_full[down_idx]

print(f"Full input feature matrix X shape: {X_full.shape}")
print(f"Full output variable y shape: {y_full.shape}")
print(f"Downsampled input feature matrix X shape: {X.shape}")
print(f"Downsampled output variable y shape: {y.shape}")

X = X_full
y = y_full

# End of Block 1-1: Load Data (FIXED)
# =========================================================================================

## Section 2. Data Partition, Preprocessing, and Define PyTorch DataLoader
---
Block 2-1. Data Partition:
- The code for partitioning the data (train/validation/test split) is fixed and should not be modified.

**Block 2-2. Data Preprocessing Strategies**

**Block 2-3. Define PyTorch DataLoader**

---

In [ ]:
# =========================================================================================
# Block 2-1: Data Partition
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import torch
from torch.utils.data import TensorDataset
import numpy as np

# Partition the data: train/val/test (60/20/20) -- as in file_context_0
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)  # 0.25 x 0.8 = 0.2

# End of Block 2-1: Data Partition
# =========================================================================================

In [ ]:
# =========================================================================================
# Block 2-2: Data Preprocessing Strategies

# Standardize input features using training set statistics
from sklearn.preprocessing import RobustScaler

input_scaler = RobustScaler()

X_train_scaled = input_scaler.fit_transform(X_train)
X_val_scaled = input_scaler.transform(X_val)
X_test_scaled = input_scaler.transform(X_test)

# Apply StandardScaler to output (fit only on training set)
epsilon = 1e-12  # To avoid log(0)
output_scaler = StandardScaler()

# Apply log transform then MinMax scaling to outputs (fit only on training set)
y_train_log = np.log(y_train + epsilon)
y_val_log = np.log(y_val + epsilon)
y_test_log = np.log(y_test + epsilon)

y_train_scaled = output_scaler.fit_transform(y_train_log.reshape(-1, 1))
y_val_scaled = output_scaler.transform(y_val_log.reshape(-1, 1))
y_test_scaled = output_scaler.transform(y_test_log.reshape(-1, 1))

# End of Block 2-2: Data Preprocessing Strategies
# =========================================================================================

In [ ]:
# =========================================================================================
# Block 2-3: Define PyTorch DataLoader
from torch.utils.data import DataLoader

# Convert to PyTorch tensors and create TensorDatasets
train_set = TensorDataset(torch.tensor(X_train_scaled, dtype=torch.float32), torch.tensor(y_train_scaled, dtype=torch.float32))
val_set = TensorDataset(torch.tensor(X_val_scaled, dtype=torch.float32), torch.tensor(y_val_scaled, dtype=torch.float32))
test_set = TensorDataset(torch.tensor(X_test_scaled, dtype=torch.float32), torch.tensor(y_test_scaled, dtype=torch.float32))

batch_size = 128
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)

val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

print(f"Train set: {X_train_scaled.shape[0]} samples")
print(f"Val set: {X_val_scaled.shape[0]} samples")
print(f"Test set: {X_test_scaled.shape[0]} samples")

# End of Block 2-3: Define PyTorch DataLoader
# =========================================================================================

## Section 3. Core Loss Modeling for Magnetic Material 3C90: Define Neural Networks
---
**Block 3-1. Define Neural Network Structure**

**Block 3-2. Tune Neural Network Structure**

---

In [ ]:
# =========================================================================================
# Block 3-1: Define Neural Network Structure

import torch
import torch.nn as nn

# 3. Model definition: Fully connected, batch-norm, parameterized hidden layers, z-score weights during training
class FeedforwardNN(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, out_dim=1, num_hidden_layers=2):
        super().__init__()
        layers = []
        # Input layer
        layers.append(nn.Linear(in_dim, hidden_dim))
        # layers.append(nn.LayerNorm(hidden_dim))
        layers.append(nn.SiLU())
        # Hidden layers
        for _ in range(num_hidden_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            # layers.append(nn.LayerNorm(hidden_dim))
            layers.append(nn.SiLU())
        # Output layer
        layers.append(nn.Linear(hidden_dim, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def zscore_weights(model):
    # Z-score (standardize) all weights and biases in-place
    with torch.no_grad():
        for name, param in model.named_parameters():
            if param.requires_grad and param.data.ndimension() > 0:
                mean = param.data.mean()
                std = param.data.std()
                if std > 0:
                    param.data.sub_(mean).div_(std)
                else:
                    param.data.sub_(mean)

# End of Block 3-1: Define Neural Network Structure
# =========================================================================================

In [ ]:
# =========================================================================================
# Block 3-2: Tune Neural Network Structure

# Training setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
in_dim = X.shape[1]
hidden_dim = 256
out_dim = y.shape[1]
num_hidden_layers = 5

model = FeedforwardNN(in_dim, hidden_dim, out_dim, num_hidden_layers).to(device)
criterion = nn.MSELoss()

# End of Block 3-2: Tune Neural Network Structure (To be Adjusted)
# =========================================================================================

## Section 4. Core Loss Modeling for Magnetic Material 3C90: Train Neural Networks
---
**Block 4-1. Define Optimization Algorithm and/or Scheduler**

**Block 4-2. Define and Implement Training Loop**

---

In [ ]:
# =========================================================================================
# Block 4-1: Define Optimization Algorithm and/or Scheduler

import torch

optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

# End of Block 4-1: Define Optimization Algorithm and/or Scheduler
# =========================================================================================

In [ ]:
# =========================================================================================
# Block 4-2: Define and Implement Training Loop

import torch
import numpy as np
import copy

# Training phase
num_epochs = 1000
best_val_loss = float('inf')
best_model_state = None
patience_cnt = 0

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        # Z-score weights after each batch
        # zscore_weights(model)
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            val_loss += loss.item() * xb.size(0)
    val_loss /= len(val_loader.dataset)

    current_lr = optimizer.param_groups[0]['lr']

    scheduler.step()

    if val_loss < (best_val_loss - 1e-6):
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        patience_cnt = 0
    else:
        patience_cnt += 1

    if patience_cnt >= 30:
        break

    if (epoch+1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.5f} - Val Loss: {val_loss:.5f} - LR: {current_lr:.6f}")

# Load best model
model.load_state_dict(best_model_state)

# End of Block 4-2: Define and Implement Training Loop
# =========================================================================================

## Section 5. Model Evaluation: Accuracy and Size
---
Block 5-1. Evaluate Model Accuracy
- This block is used to evaluate and export model accuracy.

Block 5-2. Evaluate Model Size
- This block is used to evaluate and export model size.

Block 5-3. Plot and Visualize Results
- Visualize the accuracy
---

In [ ]:
# =========================================================================================
# Block 5-1: Evaluate Model Accuracy

import pandas as pd

def evaluate_model_accuracy(loader, scaler_y, model, device, set_name="Set"):
    """
    Evaluates the regression performance of the model on a given dataset loader.

    Args:
        loader: DataLoader for the dataset.
        scaler_y: Scaler used for the target variable (output_scaler).
        model: Trained model.
        device: Device to run the model on.
        set_name (str): Name of the dataset (for display purposes).

    Returns:
        mae (float): Mean Absolute Error.
        rmse (float): Root Mean Squared Error.
        mape (float): Mean Absolute Percentage Error.
        size (int): Number of samples in the dataset.
        y_true (np.ndarray): True target values (original scale).
        y_pred (np.ndarray): Predicted target values (original scale).
    """
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            preds = model(xb).cpu().numpy()
            y_pred.append(preds)
            y_true.append(yb.cpu().numpy())
    y_true = np.vstack(y_true)
    y_pred = np.vstack(y_pred)
    # Inverse transform to original y scale (log + minmax)
    y_true_log = scaler_y.inverse_transform(y_true)
    y_pred_log = scaler_y.inverse_transform(y_pred)
    y_true_orig = np.exp(y_true_log)
    y_pred_orig = np.exp(y_pred_log)
    mae = np.mean(np.abs(y_true_orig - y_pred_orig))
    rmse = np.sqrt(np.mean((y_true_orig - y_pred_orig) ** 2))
    # Compute MAPE, avoid division by zero
    epsilon = 1e-8
    mape = np.mean(np.abs((y_true_orig - y_pred_orig) / (y_true_orig + epsilon))) * 100
    size = y_true_orig.shape[0]
    return mae, rmse, mape, size, y_true_orig, y_pred_orig

# Evaluate the regression model on train, val, and test sets
model.eval()

results = []

# Train set
train_mae, train_rmse, train_mape, train_size, y_train_true, y_train_pred = evaluate_model_accuracy(
    train_loader, output_scaler, model, device, set_name="Train Set"
)
results.append({
    "Set": "Train",
    "Size": train_size,
    "MAE": train_mae,
    "RMSE": train_rmse,
    "MAPE (%)": train_mape
})

# Validation set
val_mae, val_rmse, val_mape, val_size, y_val_true, y_val_pred = evaluate_model_accuracy(
    val_loader, output_scaler, model, device, set_name="Validation Set"
)
results.append({
    "Set": "Validation",
    "Size": val_size,
    "MAE": val_mae,
    "RMSE": val_rmse,
    "MAPE (%)": val_mape
})

# Test set
test_mae, test_rmse, test_mape, test_size, y_test_true, y_test_pred = evaluate_model_accuracy(
    test_loader, output_scaler, model, device, set_name="Test Set"
)
results.append({
    "Set": "Test",
    "Size": test_size,
    "MAE": test_mae,
    "RMSE": test_rmse,
    "MAPE (%)": test_mape
})

# Create a DataFrame to store the results
metrics_df = pd.DataFrame(results, columns=["Set", "Size", "MAE", "RMSE", "MAPE (%)"])
print("\n=== REGRESSION METRICS SUMMARY TABLE ===")
print(metrics_df)

print()
print("Test MAE:")
print("Compared to mean (160,000): {:.2f}%".format(test_mae/160000*100))
print("Compared to median (41,000): {:.2f}%".format(test_mae/41000*100))
print("Compared to max (~2,800,000): {:.2f}%".format(test_mae/2800000*100))

# End of Block 5-1: Evaluate Model Accuracy
# =========================================================================================

In [ ]:
# =========================================================================================
# Block 5-2: Evaluate Model Size

# Evaluate and print the number of parameters in the model
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

num_params = count_parameters(model)
print(f"\n=== REGRESSION MODEL SIZE ===")
print(f"Number of trainable parameters: {num_params}")

# End of Block 5-2: Evaluate Model Size
# =========================================================================================

In [ ]:
# =========================================================================================
# Block 5-3: Plot and Visualize Results

import matplotlib.pyplot as plt

# Use already computed y_test_true and y_test_pred from Block 5-1
plt.figure(figsize=(8, 4))
plt.scatter(y_test_true, y_test_pred, alpha=0.7)
min_val = min(y_test_true.min(), y_test_pred.min())
max_val = max(y_test_true.max(), y_test_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--')
plt.xlabel("Ground Truth Volumetric Losses [Wm-3]")
plt.ylabel("Predicted Volumetric Losses [Wm-3]")
plt.title("Prediction vs Ground Truth (Test Set)")
plt.tight_layout()

plt.show()

# End of Block 5-3: Plot and Visualize Results
# =========================================================================================